# Sales Potential Estimation & Analysis Project

This notebook performs the steps of environment setup, raw data retrieval from Kaggle, data distribution exploration, data quality analysis (missing values, outliers), and preprocessing (data cleaning) for the **Olist (Brazil)** e-commerce dataset.

## 1. Environment Setup & Library Imports
Load data analysis libraries (`pandas`, `numpy`), visualization libraries (`matplotlib`, `seaborn`), and necessary system configurations.


In [16]:
# ===========================
# FULL NOTEBOOK SETUP
# ===========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

import sys
import os
import shutil
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

from src.utils.system_utils import (
    project_root,
    print_section_header
)


ImportError: cannot import name 'get_web_endpoint' from 'kagglesdk.kaggle_env' (D:\FPT_Syllabus\DAP301m\DAP391m_AI2013_G7\.venv\lib\site-packages\kagglesdk\kaggle_env.py)

---
# PROJECT INTRODUCTION
---


## 1.1 Business Context

Market expansion is a decision with significant financial risk due to high real estate and operational costs. Traditional approaches based on intuition or basic demographic data can easily lead to mistakes in site selection. Businesses need to leverage online shopping behavior data (E-commerce) to accurately identify the actual "demand density" of customers, thereby optimizing offline physical infrastructure allocation.



## 1.2 Problem Statement

This project uses the **Olist (Brazil)** e-commerce dataset as the central dataset for the experimental phase. The problem is: How to aggregate online transaction data (customer location, revenue, shipping costs, delivery time) combined with external demographic data (population, income) to build a **Sales Potential Estimation** model for each region, thereby providing a scientific basis for business expansion decisions.



## 1.3 Project Objectives

-   **Regional Growth Analysis:** Identify regions with the fastest online sales growth rates to pinpoint opportunities.
-   **Identify Influencing Factors:** Build regression models to clarify which factors (population, logistics costs, shopping behavior) have the strongest impact on sales.
-   **Expansion Priority Ranking:** Develop an Opportunity Score index to rank potential regions and recommend the **Top 3 optimal locations** for the site development team.


---
# DATA DESCRIPTION
---


## 2.1 Data Retrieval


In [ ]:
print_section_header("FETCH DATA FROM KAGGLE")


olist_raw_dir = project_root / "data/raw/olist/"

downloaded_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")


for file_name in os.listdir(downloaded_path):
    downloaded_file = os.path.join(downloaded_path, file_name)
    olist_dataset_file = os.path.join(olist_raw_dir, file_name)
    
    if os.path.isfile(downloaded_file):
        shutil.copy(downloaded_file, olist_dataset_file)
        print(f"Saved: {olist_dataset_file}")


## 2.2 Data Exploration


In [ ]:
olist_customers = pd.read_csv(olist_raw_dir / 'olist_customers_dataset.csv')
olist_geolocation = pd.read_csv(olist_raw_dir / 'olist_geolocation_dataset.csv')
olist_orders = pd.read_csv(olist_raw_dir / 'olist_orders_dataset.csv')
olist_order_items = pd.read_csv(olist_raw_dir / 'olist_order_items_dataset.csv')
olist_order_payments = pd.read_csv(olist_raw_dir / 'olist_order_payments_dataset.csv')
olist_order_reviews = pd.read_csv(olist_raw_dir / 'olist_order_reviews_dataset.csv')
olist_products = pd.read_csv(olist_raw_dir / 'olist_products_dataset.csv')
olist_sellers = pd.read_csv(olist_raw_dir / 'olist_sellers_dataset.csv')


dict_olist_df = {
    'Customers'      : olist_customers,
    'Geolocation'    : olist_geolocation,
    'Orders'         : olist_orders,
    'Order Items'    : olist_order_items,
    'Order Payments' : olist_order_payments,
    'Order Reviews'  : olist_order_reviews,
    'Products'       : olist_products,
    'Sellers'        : olist_sellers,
}

for name, df in dict_olist_df.items():
    print_section_header("TABLE " + name.upper())
    
    display(df.head())

    print("--- Shape ---")
    display(df.shape)
    
    print("--- Description ---")
    display(df.describe())


---
# DATA PREPROCESSING
---


## 3.1 Tasks to Perform

- Handle duplicates
- Handle Null/NaN values
- Standardize data types
- Check categories
- Invalid data
- Remove outliers

Which features to process?


## 3. Data Preprocessing

In this section, we perform:
1. Inspect raw data structure.
2. Identify data quality issues (Missing Values, Outliers).
3. Visualize distributions of key numerical variables.
4. Perform data cleaning and save the cleaned version for subsequent analysis steps.

### 3.1 Initialize target directory for processed data storage


In [ ]:
processed_olist_dir = project_root / "data/processed/olist/"

### 3.2 Overview of the 8 Raw Data Tables
Print the shape (`shape`), first 3 rows (`head(3)`) and descriptive statistics (`describe`) to get an overview of the raw data format.


In [ ]:
# ============================================================
# Print head() + describe() for each table
# ============================================================
for name, df in dict_olist_df.items():
    print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
    display(df.head(3))
    display(df.describe(include='all').T)


### 3.3 Missing Values Analysis by Table
Aggregate all columns with Null/NaN values across all 8 raw data tables to identify columns that need to be cleaned or removed.


In [ ]:
# ============================================================
# MISSING VALUES SUMMARY TABLE
# ============================================================
# NOTE: Aggregate null counts and percentages for all 8 tables
#       into a single DataFrame for easy viewing.

missing_summary = []

for name, df in dict_olist_df.items():
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    for col in df.columns:
        if null_counts[col] > 0:
            missing_summary.append({
                'Table'          : name,
                'Column'         : col,
                'Null Count'     : int(null_counts[col]),
                'Null Rate (%)'  : float(null_pct[col]),
            })

df_missing = pd.DataFrame(missing_summary).sort_values('Null Rate (%)', ascending=False)
print(f'Total columns with null values: {len(df_missing)}')
display(df_missing)


### 3.4 Data Quality Visualization
Draw charts comparing the highest Null rates across tables and the top 10 columns with the most missing values to develop an appropriate handling strategy.


In [ ]:
# ============================================================
# CHART: Null value heatmap by table & column
# ============================================================
# NOTE: Light-colored cells = has null. Helps quickly identify
#       which tables and columns need priority handling.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Subplot 1: Null rate by table ---
null_by_table = (
    df_missing
    .groupby('Table')['Null Rate (%)']
    .max()
    .sort_values(ascending=False)
)

bars = axes[0].barh(null_by_table.index, null_by_table.values,
                    color=sns.color_palette('Reds_r', len(null_by_table)))
axes[0].set_xlabel('Max Null Rate (%)')
axes[0].set_title('Highest NULL Rate by Table')
for bar, val in zip(bars, null_by_table.values):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

# --- Subplot 2: Null row count by column ---
top_null = df_missing.nlargest(10, 'Null Count')
axes[1].barh(
    top_null['Table'] + ' · ' + top_null['Column'],
    top_null['Null Count'],
    color=sns.color_palette('Blues_r', len(top_null))
)
axes[1].set_xlabel('Null Row Count')
axes[1].set_title('Top 10 Columns with Most NULLs')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.suptitle('  Data Quality Analysis — Missing Values', y=1.02, fontsize=14, fontweight='bold')
plt.show()


### 3.5 Distribution of Price (`price`) & Shipping Cost (`freight_value`)
Using histograms and boxplots to check skewness and detect outliers.


In [ ]:
# ============================================================
# CHART: Price & Shipping Cost Distribution
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- 1) Original price histogram ---
sns.histplot(olist_order_items['price'], bins=10, ax=axes[0][0])
axes[0][0].set_yscale('log')
axes[0][0].set_title('Price Distribution (All Data)')
axes[0][0].set_xlabel('Price (BRL)')
axes[0][0].set_ylabel('Count')
axes[0][0].legend()

# --- 2) Shipping cost histogram ---
axes[0][1].set_yscale('log')
sns.histplot(olist_order_items['freight_value'], bins=10, ax=axes[0][1])
axes[0][1].set_title('Shipping Cost Distribution')
axes[0][1].set_xlabel('Price (BRL)')
axes[0][1].set_ylabel('Count')

# --- 3) Price boxplot ---
sns.boxplot(y=olist_order_items['price'], ax=axes[1][0])
axes[1][0].set_title('Price Boxplot')
axes[1][0].set_ylabel('Price (BRL)')
axes[1][0].legend()

# --- 4) Shipping cost boxplot ---
sns.boxplot(y=olist_order_items['freight_value'], ax=axes[1][1])
axes[1][1].set_title('Shipping Cost Boxplot (freight_value)')
axes[1][1].set_ylabel('Shipping Cost (BRL)')

plt.tight_layout()
plt.suptitle('Price & Shipping Cost Distribution', y=1.02, fontsize=14, fontweight='bold')
plt.show()


### 3.6 Log Transformation for Highly Skewed Numerical Variables
Since the distributions of `price` and `freight_value` are highly skewed (most values concentrated in the low range but with extremely high outliers), we apply the $\log(x + 1)$ transformation to make the distribution approximate a normal distribution, which better supports subsequent regression models.


In [ ]:
# ============================================================
# CHART: Price & Shipping Cost Distribution (Log Transformed)
# ============================================================
import numpy as np

# Log transform (log1p to avoid log(0))
price_log   = np.log1p(olist_order_items['price'])
freight_log = np.log1p(olist_order_items['freight_value'])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# --- 1) Price histogram (log) ---
sns.histplot(price_log, bins=80, kde=True, ax=axes[0][0])
axes[0][0].set_title('Price Distribution (Log Scale)')
axes[0][0].set_xlabel('log(Price + 1)')
axes[0][0].set_ylabel('Count')

# --- 2) Shipping cost histogram (log) ---
sns.histplot(freight_log, bins=80, kde=True, ax=axes[0][1])
axes[0][1].set_title('Shipping Cost Distribution (Log Scale)')
axes[0][1].set_xlabel('log(Shipping Cost + 1)')
axes[0][1].set_ylabel('Count')

# --- 3) Price boxplot (log) ---
sns.boxplot(y=price_log, ax=axes[1][0])
axes[1][0].set_title('Price Boxplot (Log Scale)')
axes[1][0].set_ylabel('log(Price + 1)')

# --- 4) Shipping cost boxplot (log) ---
sns.boxplot(y=freight_log, ax=axes[1][1])
axes[1][1].set_title('Shipping Cost Boxplot (Log Scale)')
axes[1][1].set_ylabel('log(Shipping Cost + 1)')

plt.tight_layout()
plt.suptitle('  Price & Shipping Cost Distribution (After Log Transform)',
             y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Comparison statistics
print('  Price (original):')
print(olist_order_items['price'].describe().round(2))
print('\n  Price (after log):')
print(price_log.describe().round(2))


### 3.7 Review Missing Values Summary Before Data Cleaning


In [ ]:
# ============================================================
# MISSING VALUES SUMMARY TABLE
# ============================================================
# NOTE: Aggregate null counts and percentages for all 8 tables
#       into a single DataFrame for easy viewing.

missing_summary = []

for name, df in dict_olist_df.items():
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    for col in df.columns:
        if null_counts[col] > 0:
            missing_summary.append({
                'Table'          : name,
                'Column'         : col,
                'Null Count'     : int(null_counts[col]),
                'Null Rate (%)'  : float(null_pct[col]),
            })

df_missing = pd.DataFrame(missing_summary).sort_values('Null Rate (%)', ascending=False)
print(f'Total columns with null values: {len(df_missing)}')
display(df_missing)


### 3.8 Cleaning the `olist_orders` Table
*   Keep only successfully delivered orders (`order_status == 'delivered'`).
*   Drop the `order_approved_at` column as it is not needed for the logistic/sales potential model.
*   Remove records missing carrier delivery date (`order_delivered_carrier_date`) and customer delivery date (`order_delivered_customer_date`).


In [ ]:
olist_orders_clean = (
    olist_orders[olist_orders['order_status'] == 'delivered']
    .drop(columns=['order_approved_at'])
    .dropna(subset=['order_delivered_carrier_date', 'order_delivered_customer_date'])
    .copy()
)
olist_orders_clean.to_csv(processed_olist_dir + 'orders.csv', index=False)
print(f'   orders: {len(olist_orders_clean):,} rows (delivered only)')


### 3.9 Cleaning the `olist_products` Table
Remove rows missing product category name (`product_category_name`) or missing product dimension/weight information.


In [ ]:
olist_products_clean = (
    olist_products
    .dropna(subset=['product_category_name',
                    'product_weight_g',
                    'product_length_cm',
                    'product_height_cm',
                    'product_width_cm'])
    .copy()
)

olist_products_clean.to_csv(processed_olist_dir + 'products.csv', index=False)
print(f'  products: {len(olist_products_clean):,} rows (null categories removed)')


### 3.10 Cleaning the `olist_order_reviews` Table
Drop the detailed review content columns (`review_comment_title`, `review_comment_message`) to reduce storage size, keeping only the review score (`review_score`) and order ID for quantitative analysis.


In [ ]:
olist_reviews_clean = (
    olist_order_reviews
    .drop(columns=['review_comment_title', 'review_comment_message'])
    .copy()
)

olist_reviews_clean.to_csv(processed_olist_dir + 'order_reviews.csv', index=False)
print(f'  order_reviews: dropped title & message columns')

### 3.11 Data Cleaning Results Evaluation
Compare row counts before and after cleaning for key tables to monitor the amount of data removed and ensure no Null values remain.


In [ ]:

print(f'  orders   : {len(olist_orders):,} → {len(olist_orders_clean):,} rows')
print(f'  products : {len(olist_products):,} → {len(olist_products_clean):,} rows')
print(f'  reviews  : {olist_order_reviews.shape[1]} cols → {olist_reviews_clean.shape[1]} cols')

print('\n  Remaining nulls:')
for name, df in [('orders', olist_orders_clean),
                 ('products', olist_products_clean),
                 ('reviews', olist_reviews_clean)]:
    n = df.isnull().sum().sum()
    print(f'  {name:12s} → {"  clean" if n == 0 else f"⚠️ {n} nulls"}')


### 3.12 Export Cleaned Data Tables to CSV
Save the `customers`, `order_payments`, `sellers` tables in cleaned format to the preprocessed data directory.


In [ ]:
olist_customers.to_csv(processed_olist_dir      + 'customers.csv',       index=False)
olist_order_payments.to_csv(processed_olist_dir + 'order_payments.csv',   index=False)
olist_sellers.to_csv(processed_olist_dir        + 'sellers.csv',          index=False)